# 01 — Data Exploration (Week 1)

**Goal:** Get oriented in the Backblaze hard drive dataset using DuckDB, without loading
the full dataset into memory. By the end of this notebook we should know:

- What columns/types we have
- How many rows / drives / days we're working with
- Which drive models actually have enough failure events to analyze
- A first-pass framing of the target question

**Note on data path:** This notebook currently points at `data/raw/sample`, a small
synthetic dataset (500 drives x 10 days) that mimics the real Backblaze schema, just
so the whole pipeline can be demonstrated without a 10GB download. When your real
Q1 2025 data is unzipped into `data/raw/`, change `DATA_DIR` below to `"../data/raw"`
(or wherever the 90 daily CSVs live) — no other code should need to change, since
DuckDB reads the whole folder as one logical table either way.


In [1]:
import sys
sys.path.append("../src")
import load_data as ld

con = ld.get_connection()

# TODO: swap this to your real data folder once you've downloaded + unzipped it, e.g. "../data/raw"
DATA_DIR = "../data/raw/sample"


## 1. Schema check
What columns do we actually have, and what types did DuckDB infer?

In [2]:
ld.schema(con, DATA_DIR)

,column_name,column_type,null,key,default,extra
0,date,DATE,YES,None,None,None
1,serial_number,VARCHAR,YES,None,None,None
2,model,VARCHAR,YES,None,None,None
3,capacity_bytes,BIGINT,YES,None,None,None
4,failure,BIGINT,YES,None,None,None
5,smart_5_raw,BIGINT,YES,None,None,None
6,smart_187_raw,BIGINT,YES,None,None,None
7,smart_197_raw,BIGINT,YES,None,None,None
8,smart_9_raw,BIGINT,YES,None,None,None


**Real dataset note:** Backblaze's actual CSVs have ~130+ SMART columns (many mostly null,
since not all drive models report all attributes). Once you point this at the real data,
re-run this cell and check which SMART columns are almost entirely null — those are candidates
to drop in Week 2 cleaning rather than something to impute.

## 2. Row count and date range
Sanity check that we're reading what we expect (all 90 files, full quarter).

In [3]:
print("Total drive-day rows:", ld.row_count(con, DATA_DIR))
ld.date_range(con, DATA_DIR)

Total drive-day rows: 4970


,min_date,max_date,n_days
0,2025-01-01,2025-01-10,10


## 3. Drive models present
How many distinct drives and rows per model? This tells us which models are even worth analyzing.

In [4]:
ld.unique_models(con, DATA_DIR)

,model,n_drives,n_rows
0,TOSHIBA MG07ACA14TA,112,1120
1,ST12000NM0007,109,1081
2,ST4000DM000,95,938
3,WDC WD40EFRX,93,921
4,HGST HMS5C4040BLE640,91,910


## 4. Failure summary by model — the key Week 1 output

This is the number that determines project scope. Failures are rare events, so a model
with only 1-2 failure events in the whole quarter isn't going to support any meaningful
statistical claim, no matter how many healthy drive-days it has.

In [5]:
ld.failure_summary(con, DATA_DIR)

,model,drive_days,failures,failure_rate_pct
0,ST12000NM0007,1081,4.0,0.3700
1,ST4000DM000,938,3.0,0.3198
2,WDC WD40EFRX,921,2.0,0.2172
3,TOSHIBA MG07ACA14TA,1120,1.0,0.0893
4,HGST HMS5C4040BLE640,910,0.0,0.0000


## 5. Scope decision: which models make the cut?

Set a minimum failure-event threshold and see what survives. Adjust `MIN_FAILURES` once
you see the real numbers — with the full 90-day quarter you'll have far more failure
events than this toy sample, so the threshold that makes sense here (5) is likely too
low for the real data. A common rule of thumb is to want at least ~20-30 failure events
per group before trusting a comparison.

In [6]:
MIN_FAILURES = 5
eligible = ld.drives_with_enough_failures(con, DATA_DIR, min_failures=MIN_FAILURES)
eligible

,model,drive_days,failures,failure_rate_pct


## 6. Target variable framing

**Decision:** Drive-level binary classification — "ever failed vs. never failed in the quarter."

This changes the **unit of analysis** from drive-day to drive. Right now the raw data
has one row per drive per day (a drive present for 90 days = 90 rows). For this framing,
each unique `serial_number` collapses into a single row:

- **label = 1** if the drive has *any* `failure = 1` record anywhere in the quarter
- **label = 0** if the drive appears in the data but never fails

**Feature snapshot rule:** since SMART attributes are daily readings, we need one
representative reading per drive, not 90. Recommended approach: use each drive's
**most recent available reading** in the quarter.
- For failed drives, "most recent" = the reading on (or just before) the failure day —
  exactly the state we care about.
- For healthy drives, "most recent" = simply their last day in the dataset (typically
  the end of the quarter, unless the drive was decommissioned/replaced early).

**Alternative considered:** aggregate stats (e.g., max value per attribute) over each
drive's full history instead of a single snapshot. Rejected for the first pass because
several SMART attributes (e.g. reallocated sector count) are cumulative, so max ≈ last
reading anyway — the single-snapshot rule is simpler and gives the same signal for
those attributes, at lower complexity. Worth revisiting in Month 4's ML project if the
snapshot approach underperforms.

**Practical implication for Week 2:** the cleaning notebook needs a
`build_drive_level_dataset()` step that does the day→drive collapse described above,
producing one row per `serial_number` with columns: `model`, `label`, and the SMART
attributes at that snapshot. This is the dataset Week 3's statistical tests will run on.


## 7. Write the working question formally

> "Among drive models with sufficient failure events (n >= [Y] failures), which SMART
> attributes — measured at each drive's most recent reading in the quarter — are most
> associated with whether that drive ever failed during Q1 2025?"

Fill in `[Y]` once Section 5's threshold is finalized against the real data, then copy
this question into the project README's "Research question" section.


## Week 1 checklist

- [x] Loaded and previewed raw data via DuckDB (no full in-memory load)
- [x] Confirmed schema and column types
- [x] Confirmed row count / date coverage matches expectation
- [ ] Chosen which drive models are in-scope (fill in Section 5 with real thresholds)
- [ ] Written the formal target variable definition (Section 6)
- [ ] Written the working research question (Section 7) — copy into README.md
